# Analysis of the Fenlang cohort GtCheck and array mapping

This code analyses the consistency between WES and array genotypes.
However, to do this we need to first analyse the consistency of the mapping itself

**Definitions**
* WES - whole-exome or whole-genome sequencing
* Array - data from array-based genotyping
* Mapping - the table contains pairs between WES ID and ARRAY ID

**The issue**:
Very often, the mapping file quality is bad. It contains non-unique IDs (for example, one WES IDs pairing with multiple ARRAY IDs, and multiple mismatches (for example, IDs present in the real data but not covered by mapping)
This code verifies mapping and dumps all revealed inconsistencies in the separate text files.

**Definition of ID sources**
* Data IDs - IDs, present in the real data: VCFs for WES, VCFs or FAM for ARRAY
* map IDs -- IDs, exiting in the mapping file

**Required input data**
* List of WES IDs from the data
* List of ARRAY IDs from data
* Pandata dataframe containing mapping.
  * The dataframe should contain at least two columns - one for WES ID, and one for ARRAY ID
  * If you have mapping information in any other form, you need to manually create the mapping table to make this code work.

In [1]:
import pandas as pd
from collections import Counter
from pathlib import Path

## Mapping consistency analysis

### Functions to move into pipeline

In [2]:
Id=str
IdList = list[Id]
IdSet = set[Id]

def check_id_uniquness(ids:IdList) -> tuple[IdSet, IdSet]:
    """
    Returns unique and non-unique IDs from the list of IDs 
    """
    
    counts = Counter(ids)
    unique = set( (key for key,count in counts.items() if count == 1 ))
    non_unique = set( (key for key,count in counts.items() if count > 1 ))
    return unique, non_unique

def dump_ids(ids_list: IdList, filename: Path):
    """
    Dumps to the text file all items from the list
    """
    with open(filename, 'w') as f:
        f.write('\n'.join(ids_list))

def report_id_uniquness(unique, non_unique, data_type, dump_file_name=None):
    print(f"{len(unique)} unique IDs in {data_type}")
    if len(non_unique) > 0 :
        print(f"{len(non_unique)} non-unique IDs in {data_type}")
        if dump_file_name is not None:
            print(f"Non-unique samples dumped to {dump_file_name}")
            dump_ids(non_unique, dump_file_name)
    else:
        print(f"All IDs in {data_type} are unique")


def check_ids_consistency(data_ids, map_ids, caption: str, dump_prefix:Path):

    data_ids_unique, data_ids_non_unique = check_id_uniquness(data_ids)
    map_ids_unique, map_ids_non_unique = check_id_uniquness(map_ids)
    
    data_ids, map_ids = set(data_ids), set(map_ids)
    
    data_no_map = data_ids - map_ids
    if len(data_no_map) > 0:
        data_no_map_name = dump_prefix + '.ids_data_not_in_map.txt'
        print(f"{len(data_no_map)} {caption} IDs from data not present in mapping: {data_no_map_name}")
        print(f"   --> Including {len(data_no_map & data_ids_unique)} unique IDs, and {len(data_no_map & data_ids_non_unique)} non-uniqie IDS")
        dump_ids(data_no_map, data_no_map_name)
        
    else:
        print(f"All {caption} data IDs present in mapping table")
    map_no_data = map_ids - data_ids
    if len(map_no_data) > 0:
        map_no_data_name = dump_prefix + '.ids_map_not_in_data.txt'
        print(f"{len(map_no_data)} {caption} IDs from map not present in data: {map_no_data_name}")
        print(f"   --> Including {len(map_no_data & map_ids_unique)} unique IDs, and {len(map_no_data & map_ids_non_unique)} non-uniqie IDS")
        dump_ids(map_no_data, map_no_data_name)
    else:
        print(f"All {caption} IDs from mapping exist in data")
    return data_no_map, map_no_data

def mapping_mark_duplicates(mapping: pd.DataFrame, wes_id_col:str="wes_id", array_id_col:str="array_id"):
    mapping = mapping.copy()
    mapping["duplicated_wes"] = mapping[wes_id_col].duplicated(keep=False)
    mapping["duplicated_array"] = mapping[array_id_col].duplicated(keep=False)
    mapping["duplicated_id"] = mapping["duplicated_wes"] | mapping["duplicated_array"]
    return mapping

def get_ids_from_mapping(mapping, wes_id_col, array_id_col):
    """
    """
    ids_wes = mapping[wes_id_col].to_list()
    ids_array = mapping[array_id_col].to_list()
    return ids_wes, ids_array  

def report_id_stats(ids: IdList, caption:str, stats_non_unique_file: Path) -> None:
    print(f"\n=== {caption} samples validation ===")
    print(f"Total {len(ids_wes_data)} IDs in {caption} data")
    ids_unique, ids_non_unique = check_id_uniquness(ids)
    report_id_uniquness(
        ids_unique, ids_non_unique, 
        caption, stats_non_unique_file
    )
    
def report_mapping_duplicated_stats(mapping, duplicated_samples_name) -> None:
    print("\n=== Marking duplicates in ID mapping ===")
    if mapping["duplicated_id"].any():
        mapping_dubl = mapping[ mapping.duplicated_id ]
        print(f"Identified {len(mapping_dubl)} samples with non-unique WES<->array mapping: {duplicated_samples_name}")
        mapping_dubl.to_csv(duplicated_samples_name)
    else:
        print("WES<->array mapping is unique")

def verify_wes2array_mapping(
    ids_wes_data, 
    ids_array_data, 
    mapping, 
    wes_id_col="wes_id", 
    array_id_col="array_id", 
    mapping_name=""
):

    print("\n=== Mapping stats ===")
    mapping = mapping.copy()
    print(f"Total: {len(mapping)} records in mapping")
    ids_wes_mapping, ids_array_mapping = get_ids_from_mapping(mapping, wes_id_col, array_id_col)
    
    ids_wes_mapping_unique, ids_wes_mapping_non_unique = check_id_uniquness(ids_wes_mapping)
    report_id_uniquness(
        ids_wes_mapping_unique, ids_wes_mapping_non_unique, 
        "WES mapping", mapping_name + '.WES_mapping_non-unique.txt'
    )
    
    ids_array_mapping_unique, ids_array_mapping_non_unique = check_id_uniquness(ids_array_mapping)
    report_id_uniquness(
        ids_array_mapping_unique, ids_array_mapping_non_unique, 
        "ARAY mapping", mapping_name + '.ARRAY_mapping_non-unique.txt'
    )
   
    print("\n=== Mapping consistency checking ===")
    check_ids_consistency(
        ids_wes_data, ids_wes_mapping, 
        "WES", mapping_name + '.WES')
    check_ids_consistency(
        ids_array_data, ids_array_mapping, 
        "ARRAY", mapping_name + '.ARRAY')

    mapping_marked_duplicated = mapping_mark_duplicates(mapping, wes_id_col, array_id_col)
    report_mapping_duplicated_stats(mapping_marked_duplicated, mapping_name + '.non-unique-pairs.csv')    
    
    return mapping
    

### Loading data IDs

In [3]:
with open("wes.samples-Fenland.txt") as fwes:
    ids_wes_data = [s.strip() for s in fwes.readlines()]

with open("array.samples-FENEX.txt") as farray:
    ids_array_data = [s.strip() for s in farray.readlines()]

report_id_stats(ids_wes_data, "WES data", "fenland-data-non-unique.WES.txt")
report_id_stats(ids_array_data, "ARRAY data", "fenland-data-non-unique.ARRAY.txt")


=== WES data samples validation ===
Total 11809 IDs in WES data data
11809 unique IDs in WES data
All IDs in WES data are unique

=== ARRAY data samples validation ===
Total 11809 IDs in ARRAY data data
11456 unique IDs in ARRAY data
All IDs in ARRAY data are unique


### Original Fenland-FENX mapping file

In [4]:
wes2array_name = "7286_sample_lookup.csv"
wes2array = pd.read_csv(wes2array_name)
wes2array_filtered = verify_wes2array_mapping(
    ids_wes_data, ids_array_data, 
    wes2array, 
    wes_id_col="Sanger sample id", array_id_col="Description",
    mapping_name=wes2array_name.removesuffix(".csv")
)


=== Mapping stats ===
Total: 11982 records in mapping
11982 unique IDs in WES mapping
All IDs in WES mapping are unique
11870 unique IDs in ARAY mapping
56 non-unique IDs in ARAY mapping
Non-unique samples dumped to 7286_sample_lookup.ARRAY_mapping_non-unique.txt

=== Mapping consistency checking ===
All WES data IDs present in mapping table
173 WES IDs from map not present in data: 7286_sample_lookup.WES.ids_map_not_in_data.txt
   --> Including 173 unique IDs, and 0 non-uniqie IDS
130 ARRAY IDs from data not present in mapping: 7286_sample_lookup.ARRAY.ids_data_not_in_map.txt
   --> Including 130 unique IDs, and 0 non-uniqie IDS
600 ARRAY IDs from map not present in data: 7286_sample_lookup.ARRAY.ids_map_not_in_data.txt
   --> Including 598 unique IDs, and 2 non-uniqie IDS

=== Marking duplicates in ID mapping ===
Identified 112 samples with non-unique WES<->array mapping: 7286_sample_lookup.non-unique-pairs.csv


### Correcting mapping IDs using information provided by Nikole

In [5]:
array2temp = pd.read_csv("Sanger_Fenland_last_plates_exomeid_descriptionid_link.txt")

In [6]:
wes2array_corrected = pd.merge(wes2array, array2temp, left_on='Description', right_on='DescriptionID', how='left')
wes2array_corrected["ArrayID"]= wes2array_corrected["ExomeRandomID"].fillna(wes2array_corrected["Description"])
wes2array_corrected.drop(labels = ["ExomeRandomID", "DescriptionID", "Description"], axis=1, inplace=True)

In [7]:
wes2array_corrected.to_csv("7286_sample_lookup_corrected.csv")

In [8]:
wes2array_corrected.head(5)

,Sanger sample id,EGA accession,ArrayID
0,Fenland13681333,EGAN00004530747,FENEXO16758
1,Fenland13681334,EGAN00004530746,FENEXO10251
2,Fenland13681335,EGAN00004530748,FENEXO11725
3,Fenland13681336,EGAN00004530749,FENEXO18211
4,Fenland13681337,EGAN00004530751,FENEXO18317


In [9]:
wes2array_corrected_dupl = verify_wes2array_mapping(
    ids_wes_data, ids_array_data,
    wes2array_corrected, 
    wes_id_col="Sanger sample id", array_id_col="ArrayID",
    mapping_name="7286_sample_lookup_corrected"
)


=== Mapping stats ===
Total: 11982 records in mapping
11982 unique IDs in WES mapping
All IDs in WES mapping are unique
11683 unique IDs in ARAY mapping
149 non-unique IDs in ARAY mapping
Non-unique samples dumped to 7286_sample_lookup_corrected.ARRAY_mapping_non-unique.txt

=== Mapping consistency checking ===
All WES data IDs present in mapping table
173 WES IDs from map not present in data: 7286_sample_lookup_corrected.WES.ids_map_not_in_data.txt
   --> Including 173 unique IDs, and 0 non-uniqie IDS
55 ARRAY IDs from data not present in mapping: 7286_sample_lookup_corrected.ARRAY.ids_data_not_in_map.txt
   --> Including 55 unique IDs, and 0 non-uniqie IDS
431 ARRAY IDs from map not present in data: 7286_sample_lookup_corrected.ARRAY.ids_map_not_in_data.txt
   --> Including 424 unique IDs, and 7 non-uniqie IDS

=== Marking duplicates in ID mapping ===
Identified 299 samples with non-unique WES<->array mapping: 7286_sample_lookup_corrected.non-unique-pairs.csv


### Checking for human typos with FENX and FENEX

In [10]:
# IDs ARRAY prefix for data
IDs_prefix_data = Counter(( s[:6] for s in ids_array_data))
IDs_prefix_data

Counter({'FENEXO': 11233, 'FENXO2': 223})

In [11]:
IDs_prefix_mapping = Counter(( s[:6] for s in wes2array["Description"].to_list()))
IDs_prefix_mapping

Counter({'FENEXO': 11883, 'FENXO2': 99})

## Analysing bcftools GtCheck results